# Map AltiKa tracks on WW3 outputs

The method is:
- Read in both AltiKa tracks and Standalone WW3-dice data.
- Map tracks onto the WW3 grid.
- Extract sea ice concentrations along the grid, incident $H_s$, $T_p$, $\bar{\theta}$, and those at the end of the track.

In [ ]:
#This cell must be in all notebooks!
#It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in 
# arguments externally using papermill (see mkfigs.sh for details)
from pathlib import Path
### USER EDIT start
esm_file = "/scratch/ps29/nd0349/access-om3/archive/WW3-standalone-ERA5-dice-2013/experiment_datastore.json"
model = Path(esm_file).parts[-4] # "MCW-ERA5" # 
experiment = Path(esm_file).parts[-2]
iaf = "ryf" not in experiment

output_frequency = "fx" # "1day" "1mon"
print(model, experiment)
dpi=300
### USER EDIT stop

import os
from matplotlib import rcParams
%matplotlib inline
rcParams["figure.dpi"]= dpi

plotfolder=f"/g/data/{os.environ['PROJECT']}/{os.environ['USER']}/access-om3-analysis-figs/"
os.makedirs(plotfolder, exist_ok=True)

 # a similar cell under this means it's being run in batch
print("ESM datastore path: ",esm_file)
print("Plot folder path: ",plotfolder)

In [ ]:
from intake import cat
from xarray import DataTree, map_over_datasets
from distributed import Client
import glob
import xarray as xr
import cf_xarray
import numpy as np
from datetime import timedelta
import cf_xarray as cfxr
import xesmf
import re
import os
import time
import intake
from tqdm.notebook import tqdm


# Plotting
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cmocean.cm as cmo
import matplotlib.lines as mlines
import cartopy.feature as cft

# Import my functions
functions_path = os.path.abspath("/home/566/nd0349/access-om3-analysis/functions")
if functions_path not in sys.path:
    sys.path.append(functions_path)
from get_files import *
from plot_settings import *
from fstd import *
from parameters import *
π = np.pi
test()

In [ ]:
client = Client(threads_per_worker=1)
client

In [ ]:
print(client.dashboard_link)

## Read in WW3 data

In [ ]:
datastore = intake.open_esm_datastore(
    f"{esm_file}", 
    columns_with_iterables=[
            "variable",
            "variable_long_name",
            "variable_standard_name",
            "variable_cell_methods",
            "variable_units",
    ] # This is important
)

datastore

In [ ]:
datastore.unique().frequency
datastore.search(frequency=output_frequency).unique().variable#[0:10]
# esm_datastore.df.frequency.unique()

In [ ]:
def available_variables(datastore):
    """Return a pandas dataframe summarising the variables in a datastore"""
    variable_columns = [col for col in datastore.df.columns if "variable" in col]
    return (
        datastore.df[variable_columns]
        .explode(variable_columns)
        .drop_duplicates()
        .set_index("variable")
        .sort_index()
    )

In [ ]:
datastore.unique()['variable']

In [ ]:
datastore_filtered = datastore.search(
    variable=["aice_m", "hi_m", "fsdrad_m", "wave_sig_ht_m", "uvel_m", "vvel_m", "uatm_m", "vatm_m"], #frequency=output_frequency, #require_all_on="path"
)
datastore_filtered.unique()
# access-om3.cice.1mon.mean.1984-01.nc

## Load in WW3 data

In [ ]:
xarray_open_kwargs = {"chunks": {"time": 12, "nx": -1, "ny": -1}}
ds_ww3 = datastore.search(variable=["EF", "HS", "ICE", "ICEF", "ICEH", "THM", "FP0", "T02", "UAX", "UAY"], require_all_on="path").to_dask(xarray_open_kwargs=xarray_open_kwargs)
files = datastore.search(variable=["EF", "HS", "ICE", "ICEF", "ICEH", "THM", "FP0", "T02", "UAX", "UAY"], require_all_on="path").unique().filename
 
clean_dates = [re.search(r"\d{4}-\d{2}-\d{2}", f).group() for f in files]
tmp_dates = pd.to_datetime(clean_dates)

grid_ds = xr.open_dataset('/g/data/vk83/configurations/inputs/access-om3/cice/grids/global.1deg/2024.05.14/grid.nc')
ds_ww3.coords['TLON'] = np.degrees(grid_ds['tlon'])
ds_ww3.coords['TLAT'] = np.degrees(grid_ds['tlat'])
ds_ww3['tarea'] = grid_ds['tarea']
ds_ww3['HTE'] = grid_ds['hte']/100 # cm to m
ds_ww3 = ds_ww3.rename({"ni": "nx", "nj": "ny"})

# coords = datastore.search(variable=["geolat", "geolon"]).to_dask().compute()
# coords = coords.fillna(0.0)
# ds_ww3 = ds_ww3.assign_coords(coords)
ds_ww3

## Load in AltiKa data

In [ ]:
def ReadInAltika(version, year=2019):
    if version == '0.6':
        month_range = range(1,13)
        df = pd.concat((pd.read_csv('/g/data/ps29/nd0349/Fraser-2024/data/v0_15/' + str(year) + ("%02d" % (month,)) + '_output_v0_6.csv') 
                        for month in tqdm(month_range, total = len(month_range), desc = "Reading in Alex's data")),
                        ignore_index=True)
        
        df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
        
    
    elif version == '0.10':
        # Version 0.10
        df_raw = pd.read_csv('data/v0_10/' + str(year) + '_all_output_v0_10.csv')
        row_temp = df_raw.loc[0,:]
        row_temp.values
        # Fill the first row with temporary data
        df = pd.DataFrame([row_temp], columns = df_raw.columns)
        
        numberRows,NumberCols = df_raw.shape
        
        for i in range(numberRows):
            row_temp = df_raw.loc[i,:]
            row = pd.DataFrame([row_temp], columns = df_raw.columns)
            if  (row['too_many_switches_flag'].values == 0) & (row['hit_continent_flag'].values == 0) & (row['ice_edge_diff_flag'].values == 0) & (row['latAtInnerMIZ'].values < row['latAtAltiKaEdge'].values) & (row['latAtInnerMIZ'].values < row['latAtMyEdge'].values) & (np.abs(row['lonAtInnerMIZ'].values - row['lonAtAltiKaEdge'].values) < 10):
                df = pd.concat([df,row])
        df.drop([0])
        df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
    elif version == '0.11':
        # Version 0.11
        df_raw = pd.read_csv('data/v0_11/' + str(year) + '_all_output_v0_11.csv')
        row_temp = df_raw.loc[0,:]
        row_temp.values
        # Fill the first row with temporary data
        df = pd.DataFrame([row_temp], columns = df_raw.columns)
        
        numberRows,NumberCols = df_raw.shape
        
        for i in range(numberRows):
            row_temp = df_raw.loc[i,:]
            row = pd.DataFrame([row_temp], columns = df_raw.columns)
            if  (row['too_many_switches_flag'].values == 0) & (row['hit_continent_flag'].values == 0) & (row['ice_edge_diff_flag'].values == 0) & (row['latAtInnerMIZ'].values < row['latAtAltiKaEdge'].values) & (row['latAtInnerMIZ'].values < row['latAtMyEdge'].values) & (np.abs(row['lonAtInnerMIZ'].values - row['lonAtAltiKaEdge'].values) < 10):
                df = pd.concat([df,row])
        df.drop([0])
        df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
    elif version == '0.12':
        # Version 0.12
        df_raw = pd.read_csv('data/v0_12/' + str(year) + '_all_output_v0_12.csv')
        row_temp = df_raw.loc[0,:]
        row_temp.values
        # Fill the first row with temporary data
        df = pd.DataFrame([row_temp], columns = df_raw.columns)
        
        numberRows,NumberCols = df_raw.shape
        
        for i in range(numberRows):
            row_temp = df_raw.loc[i,:]
            row = pd.DataFrame([row_temp], columns = df_raw.columns)
            if  (row['too_many_switches_flag'].values == 0) & (row['hit_continent_flag'].values == 0) & (row['ice_edge_diff_flag'].values == 0) & (row['latAtInnerMIZ'].values < row['latAtAltiKaEdge'].values) & (row['latAtInnerMIZ'].values < row['latAtMyEdge'].values) & (np.abs(row['lonAtInnerMIZ'].values - row['lonAtAltiKaEdge'].values) < 10):
                df = pd.concat([df,row])
        df.drop([0])
        df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
        
    elif version == '0.15':
        # Version 0.15
        df_raw = pd.read_csv('/g/data/ps29/nd0349/Fraser-2024/data/v0_15/' + str(year) + '_all_output_v0_15.csv')
        row_temp = df_raw.loc[0,:]
        row_temp.values
        # Fill the first row with temporary data
        df = pd.DataFrame([row_temp], columns = df_raw.columns)
        
        numberRows,NumberCols = df_raw.shape
        
        for i in range(numberRows):
            row_temp = df_raw.loc[i,:]
            row = pd.DataFrame([row_temp], columns = df_raw.columns)
            if  (row['too_many_switches_flag'].values == 0) & (row['hit_continent_flag'].values == 0) & (row['ice_edge_diff_flag'].values == 0) & (row['latAtInnerMIZ'].values < row['latAtAltiKaEdge'].values) & (row['latAtInnerMIZ'].values < row['latAtMyEdge'].values) & (np.abs(row['lonAtInnerMIZ'].values - row['lonAtAltiKaEdge'].values) < 10):
                df = pd.concat([df,row])
        df.drop([0])
        
    # Add dates to dataframe
    df['date'] = pd.to_datetime(df["first_meas_time"])#, format='%Y-%m-%d').dt.round("d")
    df['day'] = pd.to_datetime(df['date']).dt.day
    df['year'] = pd.to_datetime(df['date']).dt.year
    df['month'] = pd.to_datetime(df['date']).dt.month
    return df

In [ ]:
from tqdm.notebook import tqdm

years = range(2013, 2024)
dfs = []

for year in tqdm(years):
    df = pd.DataFrame(ReadInAltika(version='0.15', year=year))
    df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
    df['date'] = pd.to_datetime(df["first_meas_time"]).dt.date # , format='%Y-%m-%d %H:%M:%S.%f').dt.date
    df['day'] = pd.to_datetime(df['date']).dt.day
    df['year'] = pd.to_datetime(df['date']).dt.year
    df['month'] = pd.to_datetime(df['date']).dt.month
    df['mizwidth_lat'] = abs(df['latAtAltiKaEdge'] - df['latAtInnerMIZ'])*111.32 # Alex's conversion from latitudes to km

    dfs.append(df)
# Combine into one dataframe
df_all = pd.concat(dfs, ignore_index=True)
df_all.head()

## Map tracks onto the WW3 grid

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from scipy.spatial import cKDTree
from tqdm.notebook import tqdm


ALTIKA_COLS = [
    "first_meas_time",
    "swhAtMyEdge",
    "lonAtMyEdge",
    "latAtMyEdge",
    "lonAtAltiKaEdge",
    "latAtAltiKaEdge",
    "lonAtInnerMIZ",
    "latAtInnerMIZ",
    "mizWidthAlongTrackFromMyEdge",
    "mizWidthAlongTrackFromAltikaEdge",
]


def lon_to_180(lon):
    return ((lon + 180) % 360) - 180


def frequency_to_period(freq):
    freq = np.asarray(freq, dtype=float)
    return np.where(freq > 0, 1 / freq, np.nan)


def haversine_km(lon1, lat1, lon2, lat2, radius=6371.0):
    lon1, lat1, lon2, lat2 = map(np.deg2rad, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return radius * 2 * np.arcsin(np.sqrt(a))


def prepare_altika_df(
    year,
    version="0.15",
    max_edge_distance_km=50,
    max_miz_width_km=500,
):
    df = pd.DataFrame(ReadInAltika(version=version, year=year))[ALTIKA_COLS].copy()

    df["first_meas_time"] = pd.to_datetime(df["first_meas_time"])
    df["date"] = df["first_meas_time"].dt.date
    df["day"] = df["first_meas_time"].dt.day
    df["year"] = df["first_meas_time"].dt.year
    df["month"] = df["first_meas_time"].dt.month

    df["altika_myedge_dist_km"] = haversine_km(
        df["lonAtAltiKaEdge"],
        df["latAtAltiKaEdge"],
        df["lonAtMyEdge"],
        df["latAtMyEdge"],
    )

    df = df[df["altika_myedge_dist_km"] < max_edge_distance_km].copy()
    df = df[df["mizWidthAlongTrackFromAltikaEdge"] < max_miz_width_km].copy()

    return df


def infer_name(ds, candidates, label):
    for name in candidates:
        if name in ds or name in ds.coords:
            return name

    raise KeyError(f"Could not infer {label}. Tried: {candidates}")


def build_ww3_grid_lookup(
    ds_ww3,
    lon_name="TLON",
    lat_name="TLAT",
    max_lat=0,
):
    lon = ds_ww3[lon_name].values
    lat = ds_ww3[lat_name].values

    if lon.ndim != 2 or lat.ndim != 2:
        raise ValueError("Expected 2D WW3 TLON/TLAT arrays")

    southern = lat <= max_lat
    y_all, x_all = np.where(southern)

    lon_rad = np.deg2rad(lon_to_180(lon[southern]))
    lat_rad = np.deg2rad(lat[southern])

    xyz = np.column_stack(
        [
            np.cos(lat_rad) * np.cos(lon_rad),
            np.cos(lat_rad) * np.sin(lon_rad),
            np.sin(lat_rad),
        ]
    )

    return {
        "tree": cKDTree(xyz),
        "y_all": y_all,
        "x_all": x_all,
        "y_dim": ds_ww3[lat_name].dims[0],
        "x_dim": ds_ww3[lat_name].dims[1],
        "lon_name": lon_name,
        "lat_name": lat_name,
        "max_lat": max_lat,
    }


def make_track_points(row, start="altika", n_points=50):
    if start == "altika":
        lon0 = row["lonAtAltiKaEdge"]
        lat0 = row["latAtAltiKaEdge"]
    elif start == "myedge":
        lon0 = row["lonAtMyEdge"]
        lat0 = row["latAtMyEdge"]
    else:
        raise ValueError("start must be 'altika' or 'myedge'")

    lon1 = row["lonAtInnerMIZ"]
    lat1 = row["latAtInnerMIZ"]

    lon0 = lon_to_180(lon0)
    lon1 = lon_to_180(lon1)

    if abs(lon1 - lon0) > 180:
        if lon0 > lon1:
            lon1 += 360
        else:
            lon0 += 360

    lons = lon_to_180(np.linspace(lon0, lon1, n_points))
    lats = np.linspace(lat0, lat1, n_points)

    return lons, lats


def nearest_grid_indices(lons, lats, grid):
    lon_rad = np.deg2rad(lon_to_180(np.asarray(lons)))
    lat_rad = np.deg2rad(np.asarray(lats))

    xyz = np.column_stack(
        [
            np.cos(lat_rad) * np.cos(lon_rad),
            np.cos(lat_rad) * np.sin(lon_rad),
            np.sin(lat_rad),
        ]
    )

    _, southern_idx = grid["tree"].query(xyz)

    return grid["y_all"][southern_idx], grid["x_all"][southern_idx]


def add_track_output_columns(
    df,
    suffix,
    ice_var,
    edge_vars,
    period_vars,
    freq_to_period_vars,
):
    output_cols = [
        f"ww3_edge_y_{suffix}",
        f"ww3_edge_x_{suffix}",
        f"ww3_inner_y_{suffix}",
        f"ww3_inner_x_{suffix}",
        f"ww3_track_grid_cell_count_{suffix}",
        f"{ice_var}_track_mean_{suffix}",
        f"{ice_var}_track_min_{suffix}",
        f"{ice_var}_track_max_{suffix}",
        f"{ice_var}_edge_{suffix}",
        f"{ice_var}_inner_{suffix}",
    ]

    for out_name in edge_vars.values():
        output_cols += [
            f"{out_name}_edge_{suffix}",
            f"{out_name}_inner_{suffix}",
        ]

    for out_name in period_vars.values():
        output_cols += [
            f"{out_name}_edge_{suffix}",
            f"{out_name}_inner_{suffix}",
        ]

    for out_name in freq_to_period_vars.values():
        output_cols += [
            f"{out_name}_edge_{suffix}",
            f"{out_name}_inner_{suffix}",
        ]

    for col in output_cols:
        df[col] = np.nan

    return df
    

def unique_grid_track(y_idx, x_idx):
    pairs = np.column_stack([y_idx, x_idx])
    _, first = np.unique(pairs, axis=0, return_index=True)
    first = np.sort(first)

    return y_idx[first], x_idx[first]


def point_value(ds_t, var_name, y, x, grid):
    value = ds_t[var_name].isel(
        {
            grid["y_dim"]: int(y),
            grid["x_dim"]: int(x),
        }
    ).values

    return float(np.asarray(value))


def track_values(ds_t, var_name, y_idx, x_idx, grid):
    values = ds_t[var_name].isel(
        {
            grid["y_dim"]: xr.DataArray(y_idx, dims="track_point"),
            grid["x_dim"]: xr.DataArray(x_idx, dims="track_point"),
        }
    ).values

    return np.asarray(values, dtype=float)

def fill_track_outputs(
    df,
    idx,
    suffix,
    fields,
    ice_var,
    edge_vars,
    period_vars,
    freq_to_period_vars,
    y_idx,
    x_idx,
):
    edge_y, edge_x = y_idx[0], x_idx[0]
    inner_y, inner_x = y_idx[-1], x_idx[-1]

    ice_track = fields[ice_var][y_idx, x_idx]

    df.loc[idx, f"ww3_edge_y_{suffix}"] = edge_y
    df.loc[idx, f"ww3_edge_x_{suffix}"] = edge_x
    df.loc[idx, f"ww3_inner_y_{suffix}"] = inner_y
    df.loc[idx, f"ww3_inner_x_{suffix}"] = inner_x
    df.loc[idx, f"ww3_track_grid_cell_count_{suffix}"] = len(y_idx)

    df.loc[idx, f"{ice_var}_track_mean_{suffix}"] = np.nanmean(ice_track)
    df.loc[idx, f"{ice_var}_track_min_{suffix}"] = np.nanmin(ice_track)
    df.loc[idx, f"{ice_var}_track_max_{suffix}"] = np.nanmax(ice_track)
    df.loc[idx, f"{ice_var}_edge_{suffix}"] = fields[ice_var][edge_y, edge_x]
    df.loc[idx, f"{ice_var}_inner_{suffix}"] = fields[ice_var][inner_y, inner_x]

    for in_var, out_name in edge_vars.items():
        df.loc[idx, f"{out_name}_edge_{suffix}"] = fields[in_var][edge_y, edge_x]
        df.loc[idx, f"{out_name}_inner_{suffix}"] = fields[in_var][inner_y, inner_x]

    for in_var, out_name in period_vars.items():
        df.loc[idx, f"{out_name}_edge_{suffix}"] = fields[in_var][edge_y, edge_x]
        df.loc[idx, f"{out_name}_inner_{suffix}"] = fields[in_var][inner_y, inner_x]

    for freq_var, out_name in freq_to_period_vars.items():
        fp_edge = fields[freq_var][edge_y, edge_x]
        fp_inner = fields[freq_var][inner_y, inner_x]

        df.loc[idx, f"{out_name}_edge_{suffix}"] = frequency_to_period(fp_edge)
        df.loc[idx, f"{out_name}_inner_{suffix}"] = frequency_to_period(fp_inner)



In [ ]:
def subset_ww3_for_mapping(
    ds_ww3,
    year,
    ice_var="ICE",
    edge_vars=None,
    period_vars=None,
    freq_to_period_vars=None,
    lon_name="TLON",
    lat_name="TLAT",
    time_name="time",
):
    if edge_vars is None:
        edge_vars = {
            "HS": "HS",
            "THM": "THM",
            "UAX": "UAX",
            "UAY": "UAY",
        }

    if period_vars is None:
        period_vars = {"T02": "TM02"}

    if freq_to_period_vars is None:
        freq_to_period_vars = {"FP0": "TP"}

    needed_vars = (
        [ice_var, lon_name, lat_name]
        + list(edge_vars.keys())
        + list(period_vars.keys())
        + list(freq_to_period_vars.keys())
    )

    needed_vars = list(dict.fromkeys(needed_vars))

    return ds_ww3[needed_vars].sel(
        {time_name: slice(f"{year}-01-01", f"{year}-12-31")}
    )


def map_altika_tracks_to_ww3_both_edges_fast(
    df_altika,
    ds_ww3,
    year,
    ice_var="ICE",
    edge_vars=None,
    period_vars=None,
    freq_to_period_vars=None,
    lon_name="TLON",
    lat_name="TLAT",
    time_name="time",
    n_points=30,
    max_lat=0,
):
    if edge_vars is None:
        edge_vars = {
            "HS": "HS",
            "THM": "THM",
            "UAX": "UAX",
            "UAY": "UAY",
        }

    if period_vars is None:
        period_vars = {"T02": "TM02"}

    if freq_to_period_vars is None:
        freq_to_period_vars = {"FP0": "TP"}

    ds_year = subset_ww3_for_mapping(
        ds_ww3,
        year=year,
        ice_var=ice_var,
        edge_vars=edge_vars,
        period_vars=period_vars,
        freq_to_period_vars=freq_to_period_vars,
        lon_name=lon_name,
        lat_name=lat_name,
        time_name=time_name,
    )

    df = df_altika.copy()
    df["first_meas_time"] = pd.to_datetime(df["first_meas_time"])

    grid = build_ww3_grid_lookup(
        ds_year,
        lon_name=lon_name,
        lat_name=lat_name,
        max_lat=max_lat,
    )

    ww3_times = pd.DatetimeIndex(pd.to_datetime(ds_year[time_name].values))
    time_idx = ww3_times.get_indexer(df["first_meas_time"], method="nearest")

    if np.any(time_idx < 0):
        raise ValueError("Could not match every AltiKa time to a WW3 time")

    df["ww3_time"] = ww3_times[time_idx]
    df["ww3_time_idx"] = time_idx

    df = add_track_output_columns(
        df,
        suffix="altikaedge",
        ice_var=ice_var,
        edge_vars=edge_vars,
        period_vars=period_vars,
        freq_to_period_vars=freq_to_period_vars,
    )

    df = add_track_output_columns(
        df,
        suffix="myedge",
        ice_var=ice_var,
        edge_vars=edge_vars,
        period_vars=period_vars,
        freq_to_period_vars=freq_to_period_vars,
    )

    vars_to_load = set([ice_var])
    vars_to_load.update(edge_vars.keys())
    vars_to_load.update(period_vars.keys())
    vars_to_load.update(freq_to_period_vars.keys())

    for ti, group in tqdm(df.groupby("ww3_time_idx"), desc="Mapping both tracks to WW3"):
        fields = {
            var: np.asarray(ds_year[var].isel({time_name: int(ti)}).values)
            for var in vars_to_load
        }

        for idx, row in group.iterrows():
            for start, suffix in [
                ("altika", "altikaedge"),
                ("myedge", "myedge"),
            ]:
                lons, lats = make_track_points(row, start=start, n_points=n_points)

                y_idx, x_idx = nearest_grid_indices(lons, lats, grid)
                y_idx, x_idx = unique_grid_track(y_idx, x_idx)

                fill_track_outputs(
                    df=df,
                    idx=idx,
                    suffix=suffix,
                    fields=fields,
                    ice_var=ice_var,
                    edge_vars=edge_vars,
                    period_vars=period_vars,
                    freq_to_period_vars=freq_to_period_vars,
                    y_idx=y_idx,
                    x_idx=x_idx,
                )

    return df

In [ ]:
# ds_ww3.variables
df_altika[["latAtAltiKaEdge", "latAtMyEdge", "latAtInnerMIZ"]].max()

In [ ]:
year = 2014

ds_ww3_year = ds_ww3.sel(
    time=slice(f"{year}-01-01", f"{year}-12-31")
)


df_altika = prepare_altika_df(year, version="0.15")

df_tracks_ww3 = map_altika_tracks_to_ww3_both_edges_fast(
    df_altika,
    ds_ww3_year,
    year=year,
    ice_var="ICE",
    edge_vars={
        "HS": "HS",
        "THM": "THM",
        "UAX": "UAX",
        "UAY": "UAY",
    },
    period_vars={"T02": "TM02"},
    freq_to_period_vars={"FP0": "TP"},
    lon_name="TLON",
    lat_name="TLAT",
    time_name="time",
    n_points=30,
    max_lat=-40,
)

df_tracks_ww3.head()


In [ ]:
out_file = f"/g/data/ps29/nd0349/Fraser-2024/data/cleaned/{year}_altika_tracks_on_ww3_both_edges.csv"

df_tracks_ww3.to_csv(out_file, index=False)

## Read in processed data

In [ ]:
df_mapped = pd.read_csv("/g/data/ps29/nd0349/Fraser-2024/data/cleaned/2013_altika_tracks_on_ww3_both_edges_n30_v0_15.csv")
df_mapped.head()

In [ ]:
df_mapped.columns

In [ ]:
plt.scatter(df_mapped["swhAtMyEdge"], df_mapped["mizWidthAlongTrackFromMyEdge"])

In [ ]:
plt.scatter(df_mapped["HS_edge_altikaedge"], df_mapped["ww3_miz_width_hs044_km_myedge"])

In [ ]:
COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)


#### Fixing the plot axes

Notice that the white land-masked regions are distorted away from the coastlines in the Arctic in the above plot. This is because a tripolar grid is used, so the grid lines are not zonal and meridional north of 65N, and consequently the nominal 1D coordinates `xh` and `yh` are incorrect. To fix this we need to use 2d coordinates `geolon` and `geolat`.

We can get these coorindates from the intake-esm datastore. However, note that `geolon` and `geolat` contain NaNs in regions where processors were masked over land. Below we replace these NaNs with zeros so that the coordinates can be used for plotting.

In [ ]:
coords = datastore.search(variable=["geolat", "geolon"]).to_dask().compute()
coords = coords.fillna(0.0)

zos = zos.assign_coords(coords)

In [ ]:
proj = ccrs.PlateCarree()
fig, ax = plt.subplots(figsize=(15,6), subplot_kw=dict(projection=proj))

zos["zos"].isel(time=-1).plot(ax=ax, x="geolon", y="geolat")

ax.coastlines()
_ = ax.gridlines()
plt.savefig(plotfolder+'exampleout.png')

In [ ]:
client.close()